# ShowPtycho: NVIDIA CUDA — basic — manual calibration

Load your local H5/QEM data, browse with Show4DSTEM, fit SSB, then review
and save the calibration. No aberration values are required.

Install the editable bobleesj GPU and Widget forks using the
[setup guide](setup/README.md). No data are downloaded.

In [ ]:
from pathlib import Path

from quantem.gpu import SSB, detector, io
import quantem.widget as qw

qw.profile()
device = "cuda:0"
source = Path("scan_master.h5")  # Your local acquisition.
save_dir = "ssb-results-cuda"

search = "broad"


# Example values: replace with your acquisition calibration.
voltage_kV = 300
semiangle_mrad = 30
scan_sampling_A = 0.5

## Load data

Keep the master, detector shards and matching microscope metadata together. The example calibration values above are placeholders. Replace all three with your acquisition values before running. These explicitly override loaded calibration.

In [ ]:
data = io.load(source, device=device)
data.calibration.voltage_kV = voltage_kV
data.calibration.semiangle_mrad = semiangle_mrad
data.calibration.scan_sampling_A = scan_sampling_A
data.calibration

## Browse with Show4DSTEM

Move across the scan to inspect diffraction patterns. Drag the virtual detector
or choose BF/ADF presets to inspect specimen regions. ShowPtycho below reuses
this same loaded acquisition; browsing does not crop or bin its data.

In [ ]:
browse = qw.Show4DSTEM(data)
browse

## Calibrate the detector

Fit the probe-disk edge to obtain detector sampling. Inspect the fitted circle
on the mean diffraction pattern before continuing.

In [ ]:
detector_fit = data.calibration.fit_detector_sampling()
detector_fit.show(size="medium", cmap="inferno")

## Detector images and mean diffraction

Show BF, DF, ADF and mean diffraction with automatic probe fitting. The default
is `size="medium"`, `columns=2`. Choose `"small"`, `"large"` or `"xlarge"` to
resize each panel. Use `rows=1` instead of `columns=2` for one row, `cmap="gray"` for grayscale, or
`save="detector-images.png"` to save the comparison. Mean diffraction uses
logarithmic contrast; the three scan images use square-root contrast.


In [ ]:
detector.show(data, size="medium", columns=2)


For individual arrays or custom detector angles, see the advanced tutorial.


## Fit and reconstruct automatically

Choose `search="standard"` for primary optics plus tilt/depth, or
`search="broad"` for exploration through fifth order followed by Newton.
Standard uses 200 trials. Broad starts with a 200-trial primary fit, then
uses 400 higher-order trials and bounded Newton refinement.

Fitting uses the native grid and all valid bright-field pixels. **Auto**
sets reconstruction sampling from the calibrated bandwidth. More fitted
terms do not by themselves establish better resolution or true probe aberrations.

A supplied rotation is retained. An estimated rotation is reviewed during
the primary fit; phase polarity alone does not independently prove rotation.

In [ ]:
ssb = SSB(data, backend="cuda")
fit = ssb.find_aberrations(search=search)
result = ssb.reconstruct(upsampling_factor="auto")
result.report()

## Review

Adjust C10, astigmatism or tilt in the widget and inspect the phase and FFT.
Auto sampling is also used by the viewer. Display contrast and phase negation
change only the view.

In [ ]:
viewer = ssb.show(show_fft=True, save_dir=save_dir)
viewer

## Save the reviewed calibration

Run this after finishing your adjustments. Reopen the same acquisition with the
same `save_dir` to restore the saved model in ShowPtycho. Use a separate output
folder for each acquisition.

In [ ]:
viewer.save_calibration()